# Landing — S&P 500 tracker prices

Source: Yahoo Finance via the `yfinance` library — not an official API, but the
established free way to read the same data Yahoo's site shows.

Four tickers, all tracking the S&P 500 in different wrappers. **SPY is the
benchmark** used in every calculation (its history goes back to 1993); the other
three exist only for a secondary "these trackers should overlap" credibility chart.

Lands **daily**, as it arrives. Collapsing to monthly is a business rule and belongs
in Silver.

In [0]:
%pip install yfinance

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import yfinance as yf

CATALOG = "`index-vs-trust-pipeline`"
TABLE = f"{CATALOG}.landing.index_prices_raw"

INDEX_TICKERS = ["SPY", "IVV", "VOO", "SPLG"]

In [0]:
pulls = []

for ticker in INDEX_TICKERS:
    print(f"downloading {ticker}...")

    # auto_adjust=False keeps the raw Close separate from the adjusted one -- we need the
    # raw Close, because the trust data has no dividends to match against.
    hist = yf.Ticker(ticker).history(period="15y", auto_adjust=False, actions=True)

    # Yahoo answers a throttled request with an empty frame, not an error, so say so here
    # rather than failing three lines later on a missing column.
    if hist.empty:
        raise RuntimeError(f"Yahoo returned no rows for {ticker} -- throttled or unreachable")

    # yfinance puts the date in the row label, not a column.
    hist = hist.reset_index()

    # Columns like "Adj Close" and "Stock Splits" have spaces, which Delta dislikes.
    hist.columns = [c.replace(" ", "_") for c in hist.columns]

    # Timestamps usually carry a US market timezone that Spark can't convert cleanly, but
    # yfinance omits it on some responses -- stripping an absent timezone is an error.
    if hist["Date"].dt.tz is not None:
        hist["Date"] = hist["Date"].dt.tz_localize(None)

    # The response never says which ETF it is, so without this column the rows are
    # unusable. Identification, not transformation.
    hist.insert(0, "ticker", ticker)

    print(f"  -> {len(hist)} rows")
    pulls.append(hist)

index_prices = pd.concat(pulls, ignore_index=True)
print(f"total {len(index_prices)} rows")
index_prices.head(3)

In [0]:
sdf = spark.createDataFrame(index_prices)
sdf.write.format("delta").mode("overwrite").saveAsTable(TABLE)

print(f"wrote {TABLE}")

## Verification

In [0]:
%sql
SELECT ticker,
       COUNT(*) AS row_count,
       MIN(Date) AS first_date,
       MAX(Date) AS last_date
FROM `index-vs-trust-pipeline`.landing.index_prices_raw
GROUP BY ticker
ORDER BY ticker;

Expect **4 tickers**, roughly 3,700-3,800 daily rows each, starting around 2011.
VOO launched Sept 2010 and SPLG's history is shorter, so their counts may differ —
that is the data, not a bug.